# MAS-SHT v10.4 — Experiment Harness (Google Colab)

Drives all comparative experiments end-to-end on Google Colab.

**Before running:**
1. **Runtime → Change runtime type → T4 GPU** (needed only for local-HF presets)
2. **Settings → Secrets** — add: `GROQ_API_KEY`, `GOOGLE_API_KEY`, `HF_API_KEY`, `TOGETHER_API_KEY`
   Missing keys only block presets that use them; Groq-based runs only need `GROQ_API_KEY`.
3. Run cells top-to-bottom. Re-running Cell 3 resumes from the last checkpoint.

## Cell 1 — Setup

In [ ]:
# === Cell 1 — Setup ============================================================
# Installs all deps, mounts Google Drive, clones the project, loads secrets.
# Safe to re-run: pip skips already-installed pkgs, Drive mount is idempotent.

import os, sys, subprocess

DEPS = [
    'openai>=1.40.0', 'google-generativeai', 'sympy', 'statsmodels',
    'bitsandbytes',                           # 4-bit quant for local-HF presets
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'datasets',
    'transformers>=4.44.0', 'accelerate', 'huggingface_hub',
    'python-dotenv', 'tabulate', 'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)

# ── Mount Google Drive (results + checkpoints persist here) ──────────────────
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — falling back to local filesystem.')

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_URL = 'https://github.com/marios4371/llm_thesis.git'
REPO_DIR = '/content/llm_thesis' if IN_COLAB else os.path.expanduser('~/llm_thesis')

if not os.path.isdir(REPO_DIR):
    print('Cloning repo...')
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    print('Clone OK.')
else:
    print('Repo already present — pulling latest...')
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


# ── Load .env file (fallback when Colab Secrets are missing) ─────────────────
# .env is searched in: repo dir → /content → cwd.
# Colab Secrets always take precedence (override=False).
from dotenv import load_dotenv
import pathlib
_env_loaded = False
for _ep in [pathlib.Path(REPO_DIR) / '.env',
            pathlib.Path('/content') / '.env',
            pathlib.Path('.') / '.env']:
    if _ep.exists():
        load_dotenv(_ep, override=False)
        print(f'Loaded .env from {_ep}')
        _env_loaded = True
        break
if not _env_loaded:
    print('No .env file found — using Colab Secrets only.')

# ── Load API keys from Colab Secrets ─────────────────────────────────────────
def _get_secret(name):
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return None
    return os.environ.get(name)

for k in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'HF_API_KEY', 'TOGETHER_API_KEY']:
    v = _get_secret(k)
    if v:
        os.environ[k] = v
    else:
        print(f'WARNING: secret {k} not set — presets that use it will fail.')

# ── Output directories (persisted on Drive) ───────────────────────────────────
ROOT           = '/content/drive/MyDrive/MAS_SHT' if IN_COLAB else os.path.expanduser('~/MAS_SHT')
RESULTS_DIR    = f'{ROOT}/results'
CHECKPOINT_DIR = f'{ROOT}/checkpoints'
ARTIFACTS_DIR  = f'{ROOT}/artifacts'
for d in [RESULTS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Repo       : {REPO_DIR}')
print(f'Results    : {RESULTS_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

## Cell 2 — Dataset

In [ ]:
# === Cell 2 — Dataset ==========================================================
# Uses EnhancedProblemManager (from Mas_solver.py) — supports 17 datasets.
#
# Grade-school level (literature reference):
#   gsm8k_test        standard GSM8K test  (★)
#   gsm-hard          larger numbers       (★★)
#   gsm-plus          8 perturbation types (★★)
#   gsm-symbolic-p2   +2 extra clauses     (★★★)
#   svamp             structural variation (★★)
#   math500           Hendrycks competition(★★★★)
#
# Uncontaminated 2025/2026 benchmarks (NEW — main thesis contribution):
#   aime_2026         AIME 2026        (★★★★★, 30 probs, integer answers, freshest)
#   aime_2025         AIME 2025        (★★★★★, 30 probs)
#   hmmt_feb_2026     HMMT Feb 2026    (★★★★★, ~35 probs, harder than AIME)
#   hmmt_feb_2025     HMMT Feb 2025    (★★★★★, ~35 probs)
#   olymmath_hard     OlymMATH EN-HARD (★★★★★, frontier ~58%)
#   amo_bench         AMO-Bench        (★★★★★, 50 original IMO problems)
#   omni_math         Omni-MATH        (★★★★★, 4428 olympiad problems)
#   livemathbench     LiveMathBench    (★★★★,  anti-contamination, monthly updates)
#
# n=30 is safe for Groq API tier (rate-limited). For local models use n=50-100.

import sys
sys.path.insert(0, REPO_DIR)
import importlib
for _m in ['Mas_solver']:
    if _m in sys.modules: del sys.modules[_m]
from dotenv import load_dotenv
import pathlib
for _ep in [pathlib.Path(REPO_DIR)/'.env', pathlib.Path('/content')/'.env',
            pathlib.Path('.')/'.env']:
    if _ep.exists(): load_dotenv(_ep, override=False); break
import Mas_solver
from Mas_solver import EnhancedProblemManager

# ── Choose datasets and sample size ──────────────────────────────────────────
# GROQ API  tier  : n=30  per dataset (rate limited ~20 MAS probs/day)
# LOCAL 7B  tier  : n=50  per dataset
# LOCAL 1.5B tier : n=100 per dataset
DATASETS   = ['gsm8k_test', 'aime_2026', 'hmmt_feb_2026', 'olymmath_hard']
N_PROBLEMS = 120              # total; ceil(120/4) = 30 per dataset
SEED       = 42

manager  = EnhancedProblemManager(random_seed=SEED)
raw_probs = manager.load_random_problems(DATASETS, N_PROBLEMS)

def _parse_answer(a):
    """Float for numeric answers; str for olympiad expressions (e.g. sqrt(3)+1)."""
    if a is None: return None
    try:    return float(a)
    except: return str(a)

PROBLEMS = [
    {
        'problem_id':  p['id'],
        'question':    p['puzzle'],
        'gold_answer': _parse_answer(p.get('answer')),
        'dataset':     p.get('dataset', 'unknown'),
    }
    for p in raw_probs
    if p.get('answer') is not None
]

print(f'Loaded {len(PROBLEMS)} problems from {DATASETS} (seed={SEED})')
from collections import Counter
for ds, cnt in Counter(p['dataset'] for p in PROBLEMS).items():
    print(f'  {ds}: {cnt}')
print('Example:', PROBLEMS[0]['question'][:120], '...')


## Optional — Reset checkpoints before a fresh run

Set `RESET_SYSTEMS` to the system names you want to restart from scratch.
Leave it as `[]` to skip (normal resume behaviour).

In [ ]:
# Optional — delete checkpoints + old CSVs for a fresh run.
# Set to [] to skip. Example: ['mas_sht_groq', 'b_pal']
import os, glob

RESET_SYSTEMS = []  # e.g. ['mas_sht_groq', 'b1_direct']

for _sys in RESET_SYSTEMS:
    _ckpt = os.path.join(CHECKPOINT_DIR, f'{_sys}.pkl')
    if os.path.exists(_ckpt):
        os.remove(_ckpt)
        print(f'Deleted checkpoint: {_ckpt}')
    for _f in glob.glob(os.path.join(RESULTS_DIR, f'{_sys}_*.csv')):
        os.remove(_f)
        print(f'Deleted CSV: {_f}')
if not RESET_SYSTEMS:
    print('RESET_SYSTEMS is empty — nothing deleted.')

## Cell 3 — Experiment Runner

Runs **B1/B2/B4/B-PAL/B-PoT** baselines + MAS-SHT full system (+ ablations if enabled).
Safe to interrupt: re-running resumes from the last checkpoint per system.

In [ ]:
# === Cell 3 — Experiment Runner ================================================
# HOW TO RESUME: just re-run this cell — completed problems are skipped.

# ── Re-load .env so freshly re-imported Mas_solver picks up API keys ────────
from dotenv import load_dotenv
import pathlib
for _ep in [pathlib.Path(REPO_DIR) / '.env',
            pathlib.Path('/content') / '.env',
            pathlib.Path('.') / '.env']:
    if _ep.exists():
        load_dotenv(_ep, override=False)
        break

# ── Reload modules after a git pull ──────────────────────────────────────────
import importlib, sys
for _mod in ['Mas_solver', 'baselines', 'evaluation_metrics']:
    if _mod in sys.modules:
        del sys.modules[_mod]

import os, time, pickle, traceback
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

import Mas_solver
from Mas_solver import (
    QualityAwarePipeline, UnifiedLLMClient, AgentRole,
    HETEROGENEOUS_PRESETS, token_budget, _extract_last_number,
)
import baselines
from baselines import (
    direct_answer, chain_of_thought, self_consistency, baseline_only,
    pal, pot,                           # [v10.4] PAL + PoT strong baselines
    BaselineResult,
)

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Pick ONE tier below by un-commenting the CONFIG block for it.
# IMPORTANT: baseline_client model must ALWAYS match the MAS preset model
#            for a fair apples-to-apples comparison.
#
# ╔══════════════════════════════════════════════════════════════════╗
# ║ TIER A — Groq API  (requires GROQ_API_KEY, ~20 problems/day)    ║
# ║ Model: qwen3-32b · No GPU needed · Hits 100K token limit fast   ║
# ╚══════════════════════════════════════════════════════════════════╝
CONFIG = {
    'baselines_to_run': ['b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot'],
    'baseline_client': {'provider': 'groq', 'model': 'qwen3-32b'},
    'mas_variants': [
        ('mas_sht_groq',      'homogeneous_groq',    True,  True),
        # ('mas_no_siv_groq', 'homogeneous_groq',    False, True),  # ablation
        # ('mas_no_sht_groq', 'homogeneous_groq',    True,  False), # ablation
    ],
    'inter_problem_delay': 1.0,
    'checkpoint_every': 5,
}

# ╔══════════════════════════════════════════════════════════════════╗
# ║ TIER B — Local 7B on T4 GPU  (FREE, needs GPU runtime)          ║
# ║ Model: Qwen2.5-Math-7B-Instruct 4-bit · No API key needed       ║
# ║ Speed: ~40s/problem · 50 problems ≈ 35 min                      ║
# ╚══════════════════════════════════════════════════════════════════╝
# CONFIG = {
#     'baselines_to_run': ['b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot'],
#     'baseline_client': {'provider': 'local_hf',
#                          'model':    'Qwen/Qwen2.5-Math-7B-Instruct',
#                          'load_4bit': True},
#     'mas_variants': [
#         ('mas_sht_qwen7b',      'qwen_math_7b_local',    True,  True),
#         # ('mas_no_siv_qwen7b', 'qwen_math_7b_local',    False, True),
#         # ('mas_no_sht_qwen7b', 'qwen_math_7b_local',    True,  False),
#     ],
#     'inter_problem_delay': 0.5,
#     'checkpoint_every': 5,
# }

# ╔══════════════════════════════════════════════════════════════════╗
# ║ TIER C — Local 1.5B on T4 GPU  (FREE, fast, weaker model)       ║
# ║ Model: Qwen2.5-Math-1.5B-Instruct · No API key needed           ║
# ║ Speed: ~8s/problem · 300 problems ≈ 40 min (full GSM8K budget)  ║
# ╚══════════════════════════════════════════════════════════════════╝
# CONFIG = {
#     'baselines_to_run': ['b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot'],
#     'baseline_client': {'provider': 'local_hf',
#                          'model':    'Qwen/Qwen2.5-Math-1.5B-Instruct'},
#     'mas_variants': [
#         ('mas_sht_tiny',      'tiny_math_homogeneous',  True,  True),
#         ('mas_sht_deepseek',  'deepseek_distill_1_5b',  True,  True),
#     ],
#     'inter_problem_delay': 0.2,
#     'checkpoint_every': 10,
# }

# ╔══════════════════════════════════════════════════════════════════╗
# ║ TIER D — Together AI  (requires TOGETHER_API_KEY, free credits) ║
# ║ Model: DeepSeek-R1-Distill-Qwen-7B · No GPU needed              ║
# ╚══════════════════════════════════════════════════════════════════╝
# CONFIG = {
#     'baselines_to_run': ['b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot'],
#     'baseline_client': {'provider': 'together',
#                          'model':    'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'},
#     'mas_variants': [
#         ('mas_sht_deepseek7b', 'small_math_homogeneous', True, True),
#     ],
#     'inter_problem_delay': 1.0,
#     'checkpoint_every': 5,
# }

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def _ckpt_path(name): return os.path.join(CHECKPOINT_DIR, f'{name}.pkl')
def _csv_path(name):  return os.path.join(RESULTS_DIR,    f'{name}_{TIMESTAMP}.csv')

def _load_ckpt(name):
    p = _ckpt_path(name)
    if os.path.isfile(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f'  checkpoint load failed ({name}): {e}')
    return []

def _save_ckpt(name, rows):
    with open(_ckpt_path(name), 'wb') as f:
        pickle.dump(rows, f)

def _is_correct(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        return abs(float(pred) - float(gold)) < 1e-3
    except (TypeError, ValueError):
        return False

def _budget_blocked(s):
    s = str(s).lower()
    return 'budget_exceeded' in s or 'rate_limit_daily' in s

def _row_from_baseline(system, preset, problem_id, gold, br: BaselineResult):
    return {
        'problem_id': problem_id, 'system': system, 'preset': preset,
        'dataset':    'gsm8k_test',
        'gold': gold, 'predicted': br.answer,
        'correct': _is_correct(br.answer, gold),
        'baseline_ans': br.answer, 'baseline_correct': _is_correct(br.answer, gold),
        'mas_used_baseline_fallback': False, 'local_hf_fallback': False,
        'error_type': br.error_type,
        'time_s': br.time_s, 'num_llm_calls': br.num_llm_calls,
        'tokens_estimated': br.tokens_estimated,
        'verification_passed': None, 'verification_confidence': None,
        'solver_agent': system,
        'siv_execution_audit_passed': None, 'siv_blueprint_answer': None,
        'siv_execution_rel_error': None, 'siv_verified': None,
        'siv_confidence': None, 'siv_givens_matched': None,
        'siv_givens_total': None, 'siv_invertible': None,
        'siv_failed_givens': '[]', 'siv_unused_givens': '[]',
        'siv_verifies_translation': False,
        'sht_triggered': False, 'sht_triage': 'n/a',
        'sht_num_candidates': 0, 'sht_api_calls': 0,
        'timestamp': datetime.now().isoformat(),
    }

def _row_from_mas(system, preset, problem_id, gold, mas_out, time_s):
    mas_block    = mas_out.get('mas', {}) or {}
    sht_block    = mas_out.get('sht', {}) or {}
    siv_block    = mas_out.get('siv', {}) or {}
    base_block   = mas_out.get('baseline', {}) or {}
    prog_metrics = mas_block.get('programmer_metrics', {}) or {}

    pred_str     = mas_block.get('answer', '')
    pred         = _extract_last_number(str(pred_str))
    base_ans_raw = base_block.get('answer', None)
    base_pred    = _extract_last_number(str(base_ans_raw)) if base_ans_raw is not None else None

    return {
        'problem_id': problem_id, 'system': system, 'preset': preset,
        'dataset':    'gsm8k_test',
        'gold': gold, 'predicted': pred,
        'correct': _is_correct(pred, gold),
        'baseline_ans': base_pred, 'baseline_correct': _is_correct(base_pred, gold),
        'mas_used_baseline_fallback': mas_block.get('used_baseline_fallback', False),
        'local_hf_fallback':          mas_block.get('local_hf_fallback', False),
        'error_type': '' if pred is not None else 'mas_returned_unknown',
        'time_s': time_s,
        'num_llm_calls': sht_block.get('api_calls_used', 3),
        'tokens_estimated': 0,
        'verification_passed':     prog_metrics.get('verification_passed', True),
        'verification_confidence': prog_metrics.get('verification_confidence', 1.0),
        'solver_agent': (mas_out.get('agents', [None])[0].agent
                         if mas_out.get('agents') else 'unknown'),
        'siv_execution_audit_passed': siv_block.get('execution_audit_passed'),
        'siv_blueprint_answer':       siv_block.get('blueprint_answer'),
        'siv_execution_rel_error':    siv_block.get('execution_rel_error'),
        'siv_verified':       siv_block.get('verified'),
        'siv_confidence':     siv_block.get('confidence'),
        'siv_givens_matched': siv_block.get('givens_matched'),
        'siv_givens_total':   siv_block.get('givens_total'),
        'siv_invertible':     siv_block.get('invertible'),
        'siv_failed_givens':  str(siv_block.get('failed_givens', [])),
        'siv_unused_givens':  str(siv_block.get('unused_givens', [])),
        'siv_verifies_translation': False,
        'sht_triggered':      sht_block.get('triggered', False),
        'sht_triage':         sht_block.get('triage_result', 'n/a'),
        'sht_num_candidates': sht_block.get('num_candidates', 0),
        'sht_api_calls':      sht_block.get('api_calls_used', 0),
        'timestamp': datetime.now().isoformat(),
    }

# ── Run baselines ─────────────────────────────────────────────────────────────
BASELINE_FNS = {
    'b1_direct':        direct_answer,
    'b2_cot':           chain_of_thought,
    'b3_sc5':           lambda c, p: self_consistency(c, p, n=5),
    'b4_baseline_only': baseline_only,
    'b_pal':            pal,        # [v10.4] PAL  (Gao et al., ICML 2023)
    'b_pot':            pot,        # [v10.4] PoT  (Chen et al., TMLR 2023)
}

if CONFIG['baselines_to_run']:
    bc = CONFIG['baseline_client']
    baseline_client = UnifiedLLMClient(provider=bc['provider'],
                                       model_override=bc['model'])
    for sys_name in CONFIG['baselines_to_run']:
        fn   = BASELINE_FNS[sys_name]
        rows = _load_ckpt(sys_name)
        done_ids = {r['problem_id'] for r in rows}
        print(f'\n[{sys_name}] resuming with {len(done_ids)}/{len(PROBLEMS)} done')
        budget_hit = False
        for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
            if p['problem_id'] in done_ids:
                continue
            try:
                br = fn(baseline_client, p['question'])
            except Exception:
                br = BaselineResult(answer=None, raw=traceback.format_exc()[-500:],
                                    num_llm_calls=0, tokens_estimated=0,
                                    time_s=0.0, error_type='exception')
            rows.append(_row_from_baseline(sys_name, bc['model'],
                                           p['problem_id'], p['gold_answer'], br))
            if _budget_blocked(br.error_type) or _budget_blocked(getattr(br, 'raw', '')[:200]):
                print(f'[{sys_name}] Groq daily budget hit at problem {i}. Resume tomorrow.')
                budget_hit = True
            if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
                _save_ckpt(sys_name, rows)
            if budget_hit:
                break
            time.sleep(CONFIG['inter_problem_delay'])
        _save_ckpt(sys_name, rows)
        df_b = pd.DataFrame(rows)
        df_b.to_csv(_csv_path(sys_name), index=False)
        n_ok = df_b['correct'].sum()
        print(f'[{sys_name}] done — {n_ok}/{len(df_b)} correct ({n_ok/len(df_b)*100:.1f}%)')

# ── Run MAS variants ──────────────────────────────────────────────────────────
for sys_name, preset, en_siv, en_sht in CONFIG['mas_variants']:
    rows = _load_ckpt(sys_name)
    done_ids = {r['problem_id'] for r in rows}
    print(f'\n[{sys_name}] preset={preset} siv={en_siv} sht={en_sht} '
          f'resuming with {len(done_ids)}/{len(PROBLEMS)} done')
    pipeline = QualityAwarePipeline(
        heterogeneous_preset=preset, use_cache=False,
        enable_siv=en_siv, enable_sht=en_sht,
        evaluation_mode=True,           # [v10.4] forces cache off during evaluation
        dataset_seed=42,                # [v10.4] reproducibility
    )
    budget_hit = False
    for i, p in enumerate(tqdm(PROBLEMS, desc=sys_name)):
        if p['problem_id'] in done_ids:
            continue
        t0 = time.time()
        try:
            mas_out = pipeline.solver.solve(p['question'], str(p['gold_answer']))
        except Exception as e:
            print(f'  exception on {p["problem_id"]}: {e}')
            mas_out = {'mas': {'answer': 'unknown'}, 'siv': {}, 'sht': {}}
        elapsed = time.time() - t0
        rows.append(_row_from_mas(sys_name, preset,
                                  p['problem_id'], p['gold_answer'], mas_out, elapsed))
        if _budget_blocked(str(mas_out)):
            print(f'[{sys_name}] daily budget hit at problem {i}. Resume tomorrow.')
            budget_hit = True
        if (i + 1) % CONFIG['checkpoint_every'] == 0 or budget_hit:
            _save_ckpt(sys_name, rows)
        if budget_hit:
            break
        time.sleep(CONFIG['inter_problem_delay'])
    _save_ckpt(sys_name, rows)
    df_m = pd.DataFrame(rows)
    df_m.to_csv(_csv_path(sys_name), index=False)
    n_ok = df_m['correct'].sum()
    print(f'[{sys_name}] done — {n_ok}/{len(df_m)} correct ({n_ok/len(df_m)*100:.1f}%)')

print('\n=== All runs complete ===')
print(token_budget.usage_report())

## Cell 4 — Aggregation

In [ ]:
# === Cell 4 — Aggregation ======================================================
# Glob all CSVs in RESULTS_DIR, dedupe (problem_id, system) keeping latest,
# expose results_dict for the metrics layer.

import glob, os
import pandas as pd

csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
print(f'Found {len(csv_paths)} CSV(s) in {RESULTS_DIR}')
if not csv_paths:
    raise SystemExit('No results yet — run Cell 3 first.')

frames = [pd.read_csv(p) for p in csv_paths]
merged = pd.concat(frames, ignore_index=True)
merged['timestamp'] = pd.to_datetime(merged.get('timestamp'), errors='coerce')
merged = (merged
          .sort_values('timestamp')
          .drop_duplicates(['problem_id', 'system'], keep='last')
          .reset_index(drop=True))

results_dict = {s: g.reset_index(drop=True) for s, g in merged.groupby('system')}

summary = (merged
           .groupby('system')
           .agg(n=('problem_id', 'nunique'),
                accuracy=('correct', 'mean'),
                avg_calls=('num_llm_calls', 'mean'),
                avg_time=('time_s', 'mean'))
           .sort_values('accuracy', ascending=False)
           .round(4))
summary['accuracy_pct'] = (summary['accuracy'] * 100).round(1).astype(str) + '%'
print(summary[['n', 'accuracy_pct', 'avg_calls', 'avg_time']].to_string())

## Cell 5 — Statistical Analysis

Wilson 95 % CI per system · paired bootstrap CI on Δ-accuracy · McNemar test.

In [ ]:
# === Cell 5 — Statistical Analysis ============================================
# Uses evaluation_metrics.py (v10.4):
#   • Wilson 95 % CI  → accuracy_ci_lo / accuracy_ci_hi
#   • Paired bootstrap CI on Δ → delta_ci_lo / delta_ci_hi
#   • McNemar (exact or Yates)  → p_value / significant_at_alpha

import sys
sys.path.insert(0, REPO_DIR)
import importlib
for _m in ['evaluation_metrics']:
    if _m in sys.modules: del sys.modules[_m]
from evaluation_metrics import compute_all_metrics, run_mcnemar_tests

# ── Auto-detect reference system ─────────────────────────────────────────────
_mas = [s for s in results_dict if s.startswith('mas_sht_')]
if not _mas:
    raise SystemExit('No mas_sht_* system found. Run Cells 3 & 4 first.')
REFERENCE = 'mas_sht_groq' if 'mas_sht_groq' in _mas else _mas[0]
print(f'Reference system : {REFERENCE}')
print(f'All systems      : {sorted(results_dict.keys())}')

# ── Full metrics table ────────────────────────────────────────────────────────
metrics_df = compute_all_metrics(results_dict, reference_system=REFERENCE)
comp_csv   = os.path.join(ARTIFACTS_DIR, 'comparison_table.csv')
metrics_df.to_csv(comp_csv, index=False)
print(f'\nSaved: {comp_csv}')

_cols = ['system', 'n_paired', 'accuracy', 'accuracy_ci_lo', 'accuracy_ci_hi',
         'delta_vs_ref', 'delta_ci_lo', 'delta_ci_hi',
         'avg_llm_calls', 'avg_time_s', 'accuracy_per_call']
print('\n=== Comparison table ===')
print(metrics_df[[c for c in _cols if c in metrics_df.columns]].to_string(index=False))

# ── McNemar tests ─────────────────────────────────────────────────────────────
mcnemar_df = run_mcnemar_tests(results_dict, reference_system=REFERENCE)
mc_csv     = os.path.join(ARTIFACTS_DIR, 'mcnemar_results.csv')
mcnemar_df.to_csv(mc_csv, index=False)
print(f'\nSaved: {mc_csv}')

_mc_cols = ['other_system', 'n_paired', 'a', 'b', 'c', 'd',
            'test_used', 'p_value', 'significant_at_alpha']
print('\n=== McNemar results ===')
print(mcnemar_df[[c for c in _mc_cols if c in mcnemar_df.columns]].to_string(index=False))

# ── Markdown table for thesis ─────────────────────────────────────────────────
try:
    from tabulate import tabulate
    _disp = ['system', 'n_paired', 'accuracy', 'accuracy_ci_lo', 'accuracy_ci_hi',
             'delta_vs_ref', 'delta_ci_lo', 'delta_ci_hi', 'avg_llm_calls']
    md_table = tabulate(
        metrics_df[[c for c in _disp if c in metrics_df.columns]],
        headers='keys', tablefmt='pipe', floatfmt='.3f', showindex=False,
    )
    md_path = os.path.join(ARTIFACTS_DIR, 'comparison_table.md')
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(md_table)
    print(f'\nSaved: {md_path}')
    print(md_table)
except ImportError:
    print('tabulate not installed — skipping Markdown table')

## Cell 6 — Plots

In [ ]:
# === Cell 6 — Plots ============================================================
# Accuracy bar chart (all systems) + SHT/SIV breakdown for MAS.
# Saves to ARTIFACTS_DIR on Drive; also shown inline in Colab.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

COLORS = {
    'b1_direct':        '#8ecae6',
    'b2_cot':           '#219ebc',
    'b4_baseline_only': '#023047',
    'b_pal':            '#6a4c93',   # [v10.4] PAL
    'b_pot':            '#9b5de5',   # [v10.4] PoT
    'mas_sht_groq':     '#f4845f',
    'mas_no_siv_groq':  '#fca17d',
    'mas_no_sht_groq':  '#fcbf49',
    'mas_sht_qwen7b':   '#e76f51',
    'mas_sht_tiny':     '#c1440e',
}
LABELS = {
    'b1_direct':        'B1: Direct',
    'b2_cot':           'B2: CoT',
    'b4_baseline_only': 'B4: Baseline\nOnly',
    'b_pal':            'B-PAL',
    'b_pot':            'B-PoT',
    'mas_sht_groq':     'MAS-SHT\n(Groq)',
    'mas_no_siv_groq':  'MAS no-SIV\n(Groq)',
    'mas_no_sht_groq':  'MAS no-SHT\n(Groq)',
    'mas_sht_qwen7b':   'MAS-SHT\nQwen-7B',
    'mas_sht_tiny':     'MAS-SHT\n1.5B',
}

_ORDER = ['b1_direct', 'b2_cot', 'b4_baseline_only', 'b_pal', 'b_pot',
          'mas_sht_tiny', 'mas_sht_qwen7b', 'mas_sht_groq',
          'mas_no_siv_groq', 'mas_no_sht_groq']
systems = [s for s in _ORDER if s in results_dict]
systems += [s for s in sorted(results_dict.keys()) if s not in systems]
accs    = [results_dict[s]['correct'].mean() for s in systems]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'MAS-SHT v10.4 — GSM8K Results (n={len(PROBLEMS)}, {len(systems)} systems)',
    fontsize=13, y=1.01)

# Panel 1: accuracy bars
ax = axes[0]
bars = ax.bar([LABELS.get(s, s) for s in systems],
              [a * 100 for a in accs],
              color=[COLORS.get(s, '#999') for s in systems],
              edgecolor='white', linewidth=1.2, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('System Accuracy Comparison', fontsize=11)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', labelsize=8, rotation=15)

# Panel 2: MAS SHT/SIV breakdown
ax2 = axes[1]
_ref = REFERENCE if 'REFERENCE' in dir() and REFERENCE in results_dict else \
       next((s for s in _ORDER if s in results_dict and s.startswith('mas_sht_')), None)
if _ref:
    df_m   = results_dict[_ref]
    total  = len(df_m)
    n_sht  = df_m['sht_triggered'].sum()
    n_nsht = total - n_sht
    sht_acc  = df_m[df_m['sht_triggered'] == True ]['correct'].mean() if n_sht  else 0.0
    nsht_acc = df_m[df_m['sht_triggered'] == False]['correct'].mean() if n_nsht else 0.0
    n_siv_t  = (df_m['siv_verified'] == True).sum()
    n_siv_f  = (df_m['siv_verified'] == False).sum()
    n_siv_na = df_m['siv_verified'].isna().sum()
    cats   = ['No SHT\n(conf. pass)', 'SHT\n(triggered)']
    vals   = [nsht_acc * 100, sht_acc * 100]
    counts = [n_nsht, n_sht]
    brs = ax2.bar(cats, vals, color=['#2a9d8f', '#e9c46a'],
                  edgecolor='white', linewidth=1.2, width=0.4)
    for bar, acc, cnt in zip(brs, vals, counts):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                 f'{acc:.1f}%\n(n={cnt})', ha='center', va='bottom', fontsize=10)
    ax2.set_ylabel('Accuracy (%)', fontsize=11)
    ax2.set_title(f'{_ref}: Accuracy by SHT Path\n'
                  f'SIV: verified={n_siv_t}, not-inv.={n_siv_f}, N/A={n_siv_na}',
                  fontsize=10)
    ax2.set_ylim(0, 110)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
else:
    ax2.text(0.5, 0.5, 'MAS data not available', ha='center', va='center',
             transform=ax2.transAxes)

plt.tight_layout()
fig_path = os.path.join(ARTIFACTS_DIR, 'comparison.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# ── Download hint ─────────────────────────────────────────────────────────────
print(f'\nAll artifacts on Google Drive at: {ARTIFACTS_DIR}')
print('Files: comparison_table.csv, comparison_table.md, mcnemar_results.csv, comparison.png')